# Pipeline

In [1]:
import argparse
import cv2
import numpy as np
import onnx
import onnxruntime as rt
from pathlib import Path
import time
from challenge.tinyyolov2 import TinyYoloV2, TinyYoloV2_BNOpt
import torch
from typing import List
from challenge.utils.viz import num_to_class
from challenge.utils.yolo import nms, filter_boxes

from challenge.utils.camera import CameraDisplay


## Load TinyYOLO

In [4]:
# load onnx model
model_path = "../models/person_only_baseline/fp16model.onnx"
model = onnx.load(model_path)
onnx.checker.check_model(model)
# create rt session
#providers = ['CUDAExecutionProvider']#,'CPUExecutionProvider']
providers = [('CUDAExecutionProvider', {"cudnn_conv_use_max_workspace": '1','cudnn_conv_algo_search': 'EXHAUSTIVE'}),'CPUExecutionProvider',]

sess_options = rt.SessionOptions()
#sess_options.enable_profiling = True
sess_options.graph_optimization_level = rt.GraphOptimizationLevel.ORT_ENABLE_EXTENDED
sess_options.optimized_model_filepath = 'optmodel.onnx'

ort_sess = rt.InferenceSession(model_path, sess_options=sess_options, providers=providers)


TinyYoloV2(
  (pad): ReflectionPad2d((0, 1, 0, 1))
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn3): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv4): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn4): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv5): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn5): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv6): Conv2d(256, 512, kernel_size=(3, 3), stride=

In [ ]:
def onnxIou(bboxes1, bboxes2):
    """ calculate iou between each bbox in `bboxes1` with each bbox in `bboxes2`"""
    #print(bboxes1[...,:4].reshape(-1, 4))
    #print(bboxes1[...,:4].reshape(-1, 4).shape)
    px, py, pw, ph = np.array_split(bboxes1[...,:4].reshape(-1, 4), 4, axis=-1)
    #print(bboxes2[...,:4].reshape(-1, 4))
    lx, ly, lw, lh = np.array_split(bboxes2[...,:4].reshape(-1, 4), 4, axis=-1)
    #print(px)
    #print(px.shape)
    #print(lx, ly, lw, lh)
    px1, py1, px2, py2 = px - 0.5 * pw, py - 0.5 * ph, px + 0.5 * pw, py + 0.5 * ph
    lx1, ly1, lx2, ly2 = lx - 0.5 * lw, ly - 0.5 * lh, lx + 0.5 * lw, ly + 0.5 * lh
    zero = np.array(0.0, dtype=px1.dtype)
    
    dx = np.maximum(np.minimum(px2, lx2.T) - np.maximum(px1, lx1.T), zero)
    dy = np.maximum(np.minimum(py2, ly2.T) - np.maximum(py1, ly1.T), zero)
    intersections = dx * dy
    pa = (px2 - px1) * (py2 - py1) # area
    la = (lx2 - lx1) * (ly2 - ly1) # area
    unions = (pa + la.T) - intersections
    ious = (intersections/unions).reshape(*bboxes1.shape[:-1], *bboxes2.shape[:-1])

    return ious

def onnxNms(filtered_ndarray: List[np.ndarray], threshold: float) -> List[np.ndarray]:
    result = []
    for x in filtered_ndarray:
        # Sort coordinates by descending confidence
        order = np.argsort(x[:, 4], 0)
        order = order[::-1]
        x = x[order]
        ious = onnxIou(x,x) # get ious between each bbox in x

        # Filter based on iou
        keep = (ious > threshold).astype(np.int_)
        keep = np.triu(keep, 1)
        keep = np.sum(keep, 0, keepdims=True)
        keep = keep.T
        keep = np.broadcast_to(keep, x.shape) == 0

        result.append(x[keep].reshape(-1, 6))
    return result

def onnxFilter_boxes(output_ndarray: np.ndarray, threshold) -> List[np.ndarray]:
    b, a, h, w, c = output_ndarray.shape
    x = output_ndarray.reshape(b, a * h * w, c)
    #print(x.shape)
    boxes = x[:, :, 0:4]
    confidence = x[:, :, 4]
    #print(x[:, :, 5:].shape)
    scores = np.max(x[:, :, 5:], axis=-1)
    idx = np.argmax(x[:, :, 5:], axis=-1)
    
    idx = np.float32(idx)
    scores = scores * confidence
    mask = scores > threshold
    
    filtered = []
    for c, s, i, m in zip(boxes, scores, idx, mask):
        if m.any():
            detected = np.concatenate((c[m, :], s[m, None], i[m, None]), -1)
        else:
            detected = np.zeros((0, 6), dtype=x.dtype)
        filtered.append(detected)
    return filtered

def onnxDisplayBoxes(frame, output: List[torch.tensor]):
    # frame shape is (320,320,3)
    pad = 20
    frame = np.pad(frame, ((pad, pad), (pad, pad), (0, 0)), mode='constant')
    img_shape = 320
        
    if output:
        bboxes = np.stack(output, axis=0)
        for i in range(bboxes.shape[1]):
            if bboxes[0,i,-1] >= 0:
                '''                
                Index meanings:
                0 - x center
                1 - y center
                2 - Width
                3 - Height
                4 - confidence
                5 - class idx
                '''                
                # top left corner
                start_point = (
                    int(bboxes[0,i,0]*img_shape - bboxes[0,i,2]*img_shape/2) - pad,
                    int(bboxes[0,i,1]*img_shape - bboxes[0,i,3]*img_shape/2) - pad
                )
                # bottom right corner
                end_point = (
                    int(bboxes[0,i,0]*img_shape + bboxes[0,i,2]*img_shape/2) + pad,
                    int(bboxes[0,i,1]*img_shape + bboxes[0,i,3]*img_shape/2) + pad
                )
                color = (255, 0, 0)  # BGR
                #print(num_to_class(int(bboxes[0,i,5])))
                #print((start_point, end_point))
                
                cv2.rectangle(frame, start_point, end_point, color, 2) 

    return frame


In [7]:
def callback(frame):
    global now

    fps = f"{int(1/(time.time() - now))}"
    now = time.time()
        
    frame = frame.astype('float32') / 255.0
        
    # run model
    input = np.array(frame).transpose(2,0,1).reshape(1,3,320,320).astype('float16')
    #print(f"reshape: {(time.time() - test)*1000:.2f} ms")
    test = time.time()
    output = ort_sess.run(['224'], {'input.1': input})
    print(f"model: \t {(time.time() - test)*1000:.2f} ms")

    
    # filter boxes based on confidence score (class_score*confidence)
    output = onnxFilter_boxes(output[0], 0.1)
    #filter boxes based on overlap
    output = onnxNms(output, 0.25)
    # add bounding boxes
    frame = onnxDisplayBoxes(frame, output)    
    
    
    #cv2.line(frame, (125, 34), (130, 221), (255,0,0), 10) 
    cv2.putText(frame, "fps="+fps, (2, 25), cv2.FONT_HERSHEY_SIMPLEX, 1,
                (100, 255, 0), 2, cv2.LINE_AA)
    
    return frame

## Camera loop

In [ ]:
# Initialize the camera with the callback
cam = CameraDisplay(callback)

In [ ]:
# The camera stream can be started with cam.start()
# The callback gets asynchronously called (can be stopped with cam.stop())
cam.start()

In [ ]:
# The camera should always be stopped and released for a new camera is instantiated (calling CameraDisplay(callback) again)
cam.stop()
cam.release()